# Project: Customer Service Bot

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) LangChain roadmap.

You will build a customer service agent with PII detection, structured ticket output, conversation memory, and human-in-the-loop refund approval.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define the Support Ticket Schema

Use a Pydantic model to structure the agent's ticket output.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum

class TicketCategory(str, Enum):
    billing = "billing"
    technical = "technical"
    refund = "refund"
    general = "general"

class SupportTicket(BaseModel):
    customer_name: str = Field(description="The customer's name")
    issue_summary: str = Field(description="Brief summary of the issue")
    category: TicketCategory = Field(description="Ticket category")
    priority: str = Field(description="Priority: low, medium, or high")
    resolution: str = Field(description="Proposed resolution or next steps")
    requires_refund: bool = Field(description="Whether a refund is needed")
    refund_amount: Optional[float] = Field(default=None, description="Refund amount if applicable")

print(SupportTicket.model_json_schema())

## 4. Create the Refund Tool

This tool processes refunds and will require human approval.

In [ ]:
from langchain_core.tools import tool

@tool
def process_refund(order_id: str, amount: float, reason: str) -> str:
    """Process a refund for a customer order. This action requires approval and cannot be undone."""
    return f"Refund of ${amount:.2f} processed for order {order_id}. Reason: {reason}"

## 5. Assemble the Agent

Combine PII middleware, memory, structured output, and tools.

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langgraph.prebuilt.middleware import PIIMiddleware
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model("gpt-4o-mini", model_provider="openai")
checkpointer = InMemorySaver()
pii_middleware = PIIMiddleware(strategy="redact")

agent = create_react_agent(
    model=model,
    tools=[process_refund],
    prompt=(
        "You are a customer service agent for an online store. "
        "Help customers with billing questions, technical issues, and refund requests. "
        "Always be polite and professional. "
        "When a customer needs a refund, use the process_refund tool. "
        "Summarize each interaction as a support ticket."
    ),
    response_format=SupportTicket,
    checkpointer=checkpointer,
    middleware=[pii_middleware],
)

## 6. Conversation with Memory

Use `thread_id` to maintain context across turns.

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "customer-alice-001"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Hi, I'm Alice. I ordered a laptop (order ORD-5521) but it arrived damaged.")]},
    config=config,
)
ticket = result["structured_response"]
print(f"Category: {ticket.category}")
print(f"Priority: {ticket.priority}")
print(f"Summary: {ticket.issue_summary}")

## 7. Follow-Up in the Same Thread

The agent remembers the previous context.

In [ ]:
result = agent.invoke(
    {"messages": [HumanMessage(content="I'd like a refund of $899 for the damaged laptop.")]},
    config=config,
)
ticket = result["structured_response"]
print(f"Requires refund: {ticket.requires_refund}")
print(f"Refund amount: ${ticket.refund_amount}")
print(f"Resolution: {ticket.resolution}")

## 8. Human-in-the-Loop for Refund Approval

Pause the agent before processing a refund and require human approval.

In [ ]:
from langgraph.types import Command

config_refund = {"configurable": {"thread_id": "refund-review-001"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="I'm Bob. Please refund $150 for order ORD-7789, the item was defective.")]},
    config=config_refund,
    interrupt_before=["tools"],
)

print("Agent paused. Inspecting pending action...")

## 9. Inspect and Approve the Refund

In [ ]:
state = agent.get_state(config_refund)

for msg in state.values["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"Pending: {tc['name']}({tc['args']})")

result = agent.invoke(Command(resume=True), config=config_refund)
ticket = result["structured_response"]
print(f"\nCustomer: {ticket.customer_name}")
print(f"Category: {ticket.category}")
print(f"Resolution: {ticket.resolution}")
print(f"Refund: ${ticket.refund_amount}")

## 10. Rejecting a Refund

In [ ]:
config_reject = {"configurable": {"thread_id": "refund-reject-001"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="I'm Carol. Refund $500 for order ORD-9999, I changed my mind.")]},
    config=config_reject,
    interrupt_before=["tools"],
)

result = agent.invoke(
    Command(resume="Refund denied. Per policy, change-of-mind refunds are not eligible. Offer store credit instead."),
    config=config_reject,
)
ticket = result["structured_response"]
print(f"Category: {ticket.category}")
print(f"Resolution: {ticket.resolution}")

## 11. PII Protection in Action

The middleware redacts sensitive data before the agent processes it.

In [ ]:
config_pii = {"configurable": {"thread_id": "pii-test-001"}}

result = agent.invoke(
    {"messages": [HumanMessage(
        content="My name is Dave, email dave@company.com, card 4111-1111-1111-1111. I need help with order ORD-3345."
    )]},
    config=config_pii,
)
ticket = result["structured_response"]
print(f"Customer: {ticket.customer_name}")
print(f"Summary: {ticket.issue_summary}")
print(f"Resolution: {ticket.resolution}")

## Summary

- Combine `PIIMiddleware`, `InMemorySaver`, `response_format`, and `interrupt_before` in one agent
- Pydantic schemas enforce structured ticket output via `response_format`
- `PIIMiddleware(strategy="redact")` protects sensitive customer data
- `InMemorySaver` with `thread_id` maintains conversation context
- `interrupt_before=["tools"]` pauses for human approval on refunds
- `Command(resume=True)` approves; pass a string to reject